In [106]:
import pandas as pd
import numpy as np

In [107]:
n2017_df = pd.read_csv('data/nba_season_2017.csv')
players_df = pd.read_csv('data/player_data.csv')

In [108]:
n2017_df.head(1)

,Year,Player,Pos,Age,Tm,G,GS,MP,PER,TS%,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,2017.0,Alex Abrines,SG,23.0,OKC,68.0,6.0,1055.0,10.1,0.56,...,0.898,18.0,68.0,86.0,40.0,37.0,8.0,33.0,114.0,406.0


In [109]:
players_df.head(1)

,name,year_start,year_end,position,height,weight,birth_date,college
0,Alaa Abdelnaby,1991,1995,F-C,6-10,240.0,"June 24, 1968",Duke University


# INDTRODUCTION 

In this project, we'll work with data from the 2017 NBA Season (a pretty exciting one) to perform Data Wrangling techniques, and then some analysis.

We'll start by performing some Data Wrangling techniques to join the data from the season with that of players. We'll then perform different modifications and cleaning tasks to make sure our data is ready for analysis.

Finally, we'll perform some analysis using Group By and Transform operations.

## Data Wrangling Activities

### Merging the data
Let's start by merging the data from the season with player's data.

#### 1. Merge `n2017_df` and `players_df` with a left join

Merge `n2017_df` and `players_df` using a left outer join, that means, we want to have all the stats information, but if there are missing values that can't be matched from `players_df`, we want to set those season values as null.

Store the results from the merge in the variable `df`.

In [110]:
n2017_df.columns = n2017_df.columns.str.strip()
players_df.columns = players_df.columns.str.strip()

In [111]:
df = n2017_df.merge(players_df, how='left', left_on='Player', right_on='name')

In [112]:
df.head()

,Year,Player,Pos,Age,Tm,G,GS,MP,PER,TS%,...,PF,PTS,name,year_start,year_end,position,height,weight,birth_date,college
0,2017.0,Alex Abrines,SG,23.0,OKC,68.0,6.0,1055.0,10.1,0.560,...,114.0,406.0,Alex Abrines,2017.0,2018.0,G-F,6-6,190.0,"August 1, 1993",NaN
1,2017.0,Quincy Acy,PF,26.0,TOT,38.0,1.0,558.0,11.8,0.565,...,67.0,222.0,Quincy Acy,2013.0,2018.0,F,6-7,240.0,"October 6, 1990",Baylor University
2,2017.0,Quincy Acy,PF,26.0,DAL,6.0,0.0,48.0,-1.4,0.355,...,9.0,13.0,Quincy Acy,2013.0,2018.0,F,6-7,240.0,"October 6, 1990",Baylor University
3,2017.0,Quincy Acy,PF,26.0,BRK,32.0,1.0,510.0,13.1,0.587,...,58.0,209.0,Quincy Acy,2013.0,2018.0,F,6-7,240.0,"October 6, 1990",Baylor University
4,2017.0,Steven Adams,C,23.0,OKC,80.0,80.0,2389.0,16.5,0.589,...,195.0,905.0,Steven Adams,2014.0,2018.0,C,7-0,255.0,"July 20, 1993",University of Pittsburgh


#### 2. Are there misses (mismatches) in the resulting dataframe?

As we performed a left outer join, if some values in from `n2017_df` couldn't be matched in `players_df`, the result will be null values (also referred as "misses" or "mismatches"). Are there any?

In [113]:
df['name'].isna().any()

np.True_

#### 3. How many rows couldn't be matched?

Based on previous activities consider `name` column to find how many misses were there in the resulting frame?

In [114]:
df['name'].isna().sum()

np.int64(4)

#### 4. Extract the names of the players that couldn't be matched

Extract the names of the players from `n2017_df` that couldn't be matched with `players_df` in a list (it must be a list), under the variable name `player_misses`.

In [115]:
df.loc[df['name'].isna()]

,Year,Player,Pos,Age,Tm,G,GS,MP,PER,TS%,...,PF,PTS,name,year_start,year_end,position,height,weight,birth_date,college
349,2017.0,Luc Mbah,SF,30.0,LAC,80.0,76.0,1787.0,10.3,0.581,...,122.0,484.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
350,2017.0,James Michael,PF,24.0,GSW,52.0,2.0,457.0,13.0,0.543,...,47.0,147.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
352,2017.0,Sheldon McClellan,SG,24.0,WAS,30.0,3.0,287.0,10.1,0.518,...,17.0,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
593,2017.0,Metta World,SF,37.0,LAL,25.0,2.0,160.0,6.2,0.380,...,18.0,57.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
player_misses = list(df.loc[df['name'].isna(), 'Player'].values)
player_misses

['Luc Mbah', 'James Michael', 'Sheldon McClellan', 'Metta World']

#### 5. Modify `players_df` with the correct names to re-try a successful merge

Now it's time to do some detective work...

We can guarantee that the players missing in `players_df` do exist, but they just have different names. Now, you must find the players discrepancies and update the names in the `players_df` dataframe.

For example, if in `n2017_df` the player's name was `"Michael J. Jordan"`, and in `players_df` it was just `"Michael Jordan"`, the task is to modify `players_df` to make it now `"Michael J. Jordan"`.

Important: Modify the `players_df` dataframe in place! If you "break" something, you can always read the data again.

In [117]:
names_mapping = {
  'Luc Mbah a Moute': 'Luc Mbah',
  'James Michael McAdoo': 'James Michael',
  'Sheldon Mac': 'Sheldon McClellan',
  'Metta World Peace': 'Metta World'
}

In [118]:
players_df.loc[players_df['name'] == 'Luc Mbah a Moute'] = 'Luc Mbah'

/var/folders/zs/6h03lndj5114078br2jdh8380000gn/T/ipykernel_2489/459488686.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Luc Mbah' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  players_df.loc[players_df['name'] == 'Luc Mbah a Moute'] = 'Luc Mbah'
/var/folders/zs/6h03lndj5114078br2jdh8380000gn/T/ipykernel_2489/459488686.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Luc Mbah' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  players_df.loc[players_df['name'] == 'Luc Mbah a Moute'] = 'Luc Mbah'
/var/folders/zs/6h03lndj5114078br2jdh8380000gn/T/ipykernel_2489/459488686.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Luc Mbah' has dtype incompatible wi

In [119]:
for new_name, name_2017 in names_mapping.items():
  players_df.loc[players_df['name'] == new_name, 'name'] = name_2017

#### 6. Perform the merge between `n2017_df` and `players_df` again, this time, without misses

Now that you've fixed the data in `players_df`, perform the merge between `n2017_df` and `players_df` again. Should be the same merge as before, left outer. Store the result in df.

In [120]:
df = n2017_df.merge(players_df, how='left', left_on='Player', right_on='name')

In [121]:
df

,Year,Player,Pos,Age,Tm,G,GS,MP,PER,TS%,...,PF,PTS,name,year_start,year_end,position,height,weight,birth_date,college
0,2017.0,Alex Abrines,SG,23.0,OKC,68.0,6.0,1055.0,10.1,0.560,...,114.0,406.0,Alex Abrines,2017,2018,G-F,6-6,190.0,"August 1, 1993",NaN
1,2017.0,Quincy Acy,PF,26.0,TOT,38.0,1.0,558.0,11.8,0.565,...,67.0,222.0,Quincy Acy,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
2,2017.0,Quincy Acy,PF,26.0,DAL,6.0,0.0,48.0,-1.4,0.355,...,9.0,13.0,Quincy Acy,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
3,2017.0,Quincy Acy,PF,26.0,BRK,32.0,1.0,510.0,13.1,0.587,...,58.0,209.0,Quincy Acy,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
4,2017.0,Steven Adams,C,23.0,OKC,80.0,80.0,2389.0,16.5,0.589,...,195.0,905.0,Steven Adams,2014,2018,C,7-0,255.0,"July 20, 1993",University of Pittsburgh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,2017.0,Cody Zeller,PF,24.0,CHO,62.0,58.0,1725.0,16.7,0.604,...,189.0,639.0,Cody Zeller,2014,2018,C-F,7-0,240.0,"October 5, 1992",Indiana University
601,2017.0,Tyler Zeller,C,27.0,BOS,51.0,5.0,525.0,13.0,0.508,...,61.0,178.0,Tyler Zeller,2013,2018,F-C,7-0,253.0,"January 17, 1990",University of North Carolina
602,2017.0,Stephen Zimmerman,C,20.0,ORL,19.0,0.0,108.0,7.3,0.346,...,17.0,23.0,Stephen Zimmerman,2017,2017,C,7-0,240.0,"September 9, 1996","University of Nevada, Las Vegas"
603,2017.0,Paul Zipser,SF,22.0,CHI,44.0,18.0,843.0,6.9,0.503,...,78.0,240.0,Paul Zipser,2017,2018,G-F,6-8,215.0,"February 18, 1994",NaN


In [122]:
df['name'].isna().sum()

np.int64(0)

#### 7. Remove unnecessary columns

We won't use some columns in our follow up analysis, so we can drop them to simplify the understanding of the data. Drop from `df` the following columns:

```
columns_to_drop = [
    "Year",
    "PER",
    "TS%",
    "3PAr",
    "FTr",
    "USG%",
    "blanl",
    "OWS",
    "DWS",
    "WS",
    "WS/48",
    "blank2",
    "OBPM",
    "DBPM",
    "BPM",
    "VORP",
    "FG%",
    "3P%",
    "eFG%",
    "FT%",
    "name",
]
```

Important: you must modify `df` in place, removing the columns directly in the same dataframe. If you think you've made a mistake, re-read the data and perform the joins again.

In [123]:
columns_to_drop = [
    "Year",
    "PER",
    "TS%",
    "3PAr",
    "FTr",
    "USG%",
    "blanl",
    "OWS",
    "DWS",
    "WS",
    "WS/48",
    "blank2",
    "OBPM",
    "DBPM",
    "BPM",
    "VORP",
    "FG%",
    "3P%",
    "eFG%",
    "FT%",
    "name",
]

In [124]:
df.drop(columns = columns_to_drop, inplace=True)

In [125]:
df

,Player,Pos,Age,Tm,G,GS,MP,ORB%,DRB%,TRB%,...,TOV,PF,PTS,year_start,year_end,position,height,weight,birth_date,college
0,Alex Abrines,SG,23.0,OKC,68.0,6.0,1055.0,1.9,7.1,4.5,...,33.0,114.0,406.0,2017,2018,G-F,6-6,190.0,"August 1, 1993",NaN
1,Quincy Acy,PF,26.0,TOT,38.0,1.0,558.0,3.9,18.0,11.0,...,21.0,67.0,222.0,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
2,Quincy Acy,PF,26.0,DAL,6.0,0.0,48.0,4.6,15.2,9.7,...,2.0,9.0,13.0,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
3,Quincy Acy,PF,26.0,BRK,32.0,1.0,510.0,3.8,18.2,11.1,...,19.0,58.0,209.0,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
4,Steven Adams,C,23.0,OKC,80.0,80.0,2389.0,13.0,15.5,14.2,...,146.0,195.0,905.0,2014,2018,C,7-0,255.0,"July 20, 1993",University of Pittsburgh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,Cody Zeller,PF,24.0,CHO,62.0,58.0,1725.0,8.6,17.3,12.9,...,65.0,189.0,639.0,2014,2018,C-F,7-0,240.0,"October 5, 1992",Indiana University
601,Tyler Zeller,C,27.0,BOS,51.0,5.0,525.0,9.2,17.0,13.2,...,20.0,61.0,178.0,2013,2018,F-C,7-0,253.0,"January 17, 1990",University of North Carolina
602,Stephen Zimmerman,C,20.0,ORL,19.0,0.0,108.0,10.8,24.9,17.6,...,3.0,17.0,23.0,2017,2017,C,7-0,240.0,"September 9, 1996","University of Nevada, Las Vegas"
603,Paul Zipser,SF,22.0,CHI,44.0,18.0,843.0,1.9,14.2,8.0,...,40.0,78.0,240.0,2017,2018,G-F,6-8,215.0,"February 18, 1994",NaN


#### 8. Rename teams to their full names

The `Tm` column contains an acronym of the team. For example, `GSW` for Golden State Warriors. Create a new column `Team` with the full name of the team. In the associated notebook, you can find a mapping to help you in the process.

In [126]:
team_mapping = {
    "OKC": "Oklahoma City Thunder",
    "DAL": "Dallas Mavericks",
    "BRK": "Brooklyn Nets",
    "SAC": "Sacramento Kings",
    "NOP": "New Orleans Pelicans",
    "MIN": "Minnesota Timberwolves",
    "SAS": "San Antonio Spurs",
    "IND": "Indiana Pacers",
    "MEM": "Memphis Grizzlies",
    "POR": "Portland Trail Blazers",
    "CLE": "Cleveland Cavaliers",
    "LAC": "Los Angeles Clippers",
    "PHI": "Philadelphia 76ers",
    "HOU": "Houston Rockets",
    "MIL": "Milwaukee Bucks",
    "NYK": "New York Knicks",
    "DEN": "Denver Nuggets",
    "ORL": "Orlando Magic",
    "MIA": "Miami Heat",
    "PHO": "Phoenix Suns",
    "GSW": "Golden State Warriors",
    "CHO": "Charlotte Hornets",
    "DET": "Detroit Pistons",
    "ATL": "Atlanta Hawks",
    "WAS": "Washington Wizards",
    "LAL": "Los Angeles Lakers",
    "UTA": "Utah Jazz",
    "BOS": "Boston Celtics",
    "CHI": "Chicago Bulls",
    "TOR": "Toronto Raptors"
}

In [127]:
df.head()

,Player,Pos,Age,Tm,G,GS,MP,ORB%,DRB%,TRB%,...,TOV,PF,PTS,year_start,year_end,position,height,weight,birth_date,college
0,Alex Abrines,SG,23.0,OKC,68.0,6.0,1055.0,1.9,7.1,4.5,...,33.0,114.0,406.0,2017,2018,G-F,6-6,190.0,"August 1, 1993",NaN
1,Quincy Acy,PF,26.0,TOT,38.0,1.0,558.0,3.9,18.0,11.0,...,21.0,67.0,222.0,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
2,Quincy Acy,PF,26.0,DAL,6.0,0.0,48.0,4.6,15.2,9.7,...,2.0,9.0,13.0,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
3,Quincy Acy,PF,26.0,BRK,32.0,1.0,510.0,3.8,18.2,11.1,...,19.0,58.0,209.0,2013,2018,F,6-7,240.0,"October 6, 1990",Baylor University
4,Steven Adams,C,23.0,OKC,80.0,80.0,2389.0,13.0,15.5,14.2,...,146.0,195.0,905.0,2014,2018,C,7-0,255.0,"July 20, 1993",University of Pittsburgh


In [128]:
df['Team'] = df['Tm'].replace(team_mapping)

In [129]:
df[['Player', 'Tm', 'Team']].head(10)

,Player,Tm,Team
0,Alex Abrines,OKC,Oklahoma City Thunder
1,Quincy Acy,TOT,TOT
2,Quincy Acy,DAL,Dallas Mavericks
3,Quincy Acy,BRK,Brooklyn Nets
4,Steven Adams,OKC,Oklahoma City Thunder
5,Arron Afflalo,SAC,Sacramento Kings
6,Alexis Ajinca,NOP,New Orleans Pelicans
7,Cole Aldrich,MIN,Minnesota Timberwolves
8,LaMarcus Aldridge,SAS,San Antonio Spurs
9,Lavoy Allen,IND,Indiana Pacers


#### 9. Convert birthday to a datetime object

The column `birth_date` is a string in the format `Month Day, Year (August 1, 1993)`. Convert the column to a datetime object.

In [130]:
df[['Player', 'birth_date']].head(10)

,Player,birth_date
0,Alex Abrines,"August 1, 1993"
1,Quincy Acy,"October 6, 1990"
2,Quincy Acy,"October 6, 1990"
3,Quincy Acy,"October 6, 1990"
4,Steven Adams,"July 20, 1993"
5,Arron Afflalo,"October 15, 1985"
6,Alexis Ajinca,"May 6, 1988"
7,Cole Aldrich,"October 31, 1988"
8,LaMarcus Aldridge,"July 19, 1985"
9,Lavoy Allen,"February 4, 1989"


In [131]:
df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')

In [132]:
df.loc[df['birth_date'].isna(), ['Player', 'birth_date']]

,Player,birth_date
349,Luc Mbah,NaT


#### 10. Delete all players from the `TOT` team

Finally, if you explore the dataset, you'll notice that there's a team `TOT`. In reality, that team doesn't exist, and it's just an aggregation for players that have switched teams in the season.

Your task is to delete all the rows that have `TOT` in the column `Tm`. Perform the modification in the `df` dataframe, in place.

In [140]:
# df_copy = df.copy()

In [141]:
df = df[df['Tm'] != 'TOT']

In [142]:
df.loc[df['Tm'] == 'TOT']

,Player,Pos,Age,Tm,G,GS,MP,ORB%,DRB%,TRB%,...,PF,PTS,year_start,year_end,position,height,weight,birth_date,college,Team


In [143]:
df.loc[df['Tm'] == 'TOT'].index

Index([], dtype='int64')

In [144]:
df.drop(df.loc[df['Tm'] == 'TOT'].index, inplace=True)

In [145]:
df.loc[df['Tm'] == 'TOT'].index

Index([], dtype='int64')

#### 11. What's the team with the most players in the league?

Count the number of players registered in each team and answer which team has the most players.

In [147]:
df['Team'].value_counts().head()

Team
New Orleans Pelicans    27
Dallas Mavericks        24
Cleveland Cavaliers     22
Philadelphia 76ers      22
Atlanta Hawks           22
Name: count, dtype: int64

#### 12. What's the team with the lowest `FG`?

What's the team with the lowest sum of field goals `(FG)`?

In [151]:
df.groupby('Team')['FG'].sum().sort_values().head(5)

Team
Dallas Mavericks     2968.0
Memphis Grizzlies    2984.0
Utah Jazz            3033.0
Charlotte Hornets    3093.0
Brooklyn Nets        3102.0
Name: FG, dtype: float64

#### 13. What's the team with the best `FG%`?

`FG%` is defined as `FG / FGA`, that is, total field goals divided by the number of attempts. What team has the best `FG`% in the league? Enter the full team name below (example, `Dallas Mavericks`).

In [152]:
df.head(1).T

,0
Player,Alex Abrines
Pos,SG
Age,23.0
Tm,OKC
G,68.0
GS,6.0
MP,1055.0
ORB%,1.9
DRB%,7.1
TRB%,4.5


In [154]:
fg_per_team = df.groupby('Team')[['FG', 'FGA']].sum()
fg_per_team

,FG,FGA
Team,,
Atlanta Hawks,3595.0,7961.0
Boston Celtics,3168.0,6978.0
Brooklyn Nets,3102.0,6987.0
Charlotte Hornets,3093.0,7000.0
Chicago Bulls,3169.0,7142.0
Cleveland Cavaliers,3311.0,7053.0
Dallas Mavericks,2968.0,6750.0
Denver Nuggets,3377.0,7194.0
Detroit Pistons,3269.0,7282.0


In [155]:
fg_per_team['FG%'] = fg_per_team['FG'] / fg_per_team['FGA']

In [158]:
fg_per_team.sort_values(by='FG%', ascending=False).head()

,FG,FGA,FG%
Team,,,
Golden State Warriors,3532.0,7140.0,0.494678
San Antonio Spurs,3470.0,7284.0,0.476387
Los Angeles Clippers,3242.0,6819.0,0.475436
Washington Wizards,3388.0,7136.0,0.474776
Milwaukee Bucks,3190.0,6737.0,0.473505


#### 14. What's the difference between the best and worst 3P shooters (by position)?

It is known that Shooting Guards (SG) are the best 3P throwers (by efficacy). The question is, what's the difference (in accuracy / efficacy) with the worst 3P throwers, always considering by position?

Note: use the position from the `Pos` column.

In [160]:
pos_3p_acc = df.groupby('Pos')[['3P', '3PA']].sum()
pos_3p_acc

,3P,3PA
Pos,,
C,1486.0,4210.0
PF,3514.0,10210.0
PG,5662.0,15761.0
SF,5638.0,16043.0
SG,7776.0,21106.0


In [162]:
pos_3p_acc['3P%'] = pos_3p_acc['3P'] / pos_3p_acc['3PA']
pos_3p_acc.sort_values(by='3P%', ascending=False).head()

,3P,3PA,3P%
Pos,,,
SG,7776.0,21106.0,0.368426
PG,5662.0,15761.0,0.359241
C,1486.0,4210.0,0.352969
SF,5638.0,16043.0,0.351431
PF,3514.0,10210.0,0.344172


In [163]:
pos_3p_acc['3P%'].max() - pos_3p_acc['3P%'].min()

np.float64(0.024253659969040164)

#### 15. Find the best scorers in each team

Create a new dataframe containing the best scorers per team (by `PTS`, total points scored). The resulting dataframe should contain the columns Player, `Team`, `Pos` and `PTS`, and should be stored in the variable `best_scorers_per_team`. It should be sorted by `PTS` in descending mode.

In [164]:
df['Best score per team'] = df.groupby('Team')['PTS'].transform('max')

In [165]:
df.loc[df['Tm'] == 'OKC', ['Player', 'Tm', 'PTS', 'Best score per team']].sort_values(by='PTS', ascending=False).head()

,Player,Tm,PTS,Best score per team
567,Russell Westbrook,OKC,2558.0,2558.0
421,Victor Oladipo,OKC,1067.0,2558.0
301,Enes Kanter,OKC,1033.0,2558.0
4,Steven Adams,OKC,905.0,2558.0
468,Andre Roberson,OKC,522.0,2558.0


In [167]:
df.loc[df['PTS'] == df['Best score per team'], ['Player', 'Team', 'Pos', 'PTS']].sort_values(by='PTS', ascending=False)

,Player,Team,Pos,PTS
567,Russell Westbrook,Oklahoma City Thunder,PG,2558.0
214,James Harden,Houston Rockets,PG,2356.0
525,Isaiah Thomas,Boston Celtics,PG,2199.0
122,Anthony Davis,New Orleans Pelicans,C,2099.0
538,Karl-Anthony Towns,Minnesota Timberwolves,C,2061.0
331,Damian Lillard,Portland Trail Blazers,PG,2024.0
130,DeMar DeRozan,Toronto Raptors,SG,2020.0
120,Stephen Curry,Golden State Warriors,PG,1999.0
274,LeBron James,Cleveland Cavaliers,SF,1954.0
324,Kawhi Leonard,San Antonio Spurs,SF,1888.0


#### 16. Which team has the 'youngest squad', by average player age?

Calculate the average player age per team and answer which team has the "youngest squad"?

In [173]:
df.groupby('Team')['birth_date'].mean().sort_values(ascending=False)

Team
Portland Trail Blazers   1992-03-14 08:00:00.000000000
Toronto Raptors          1991-04-16 16:56:28.235294080
Boston Celtics           1991-04-04 19:12:00.000000000
Orlando Magic            1991-01-31 16:25:15.789473664
Denver Nuggets           1991-01-14 17:41:03.157894784
Detroit Pistons          1991-01-13 17:36:00.000000000
Phoenix Suns             1991-01-01 02:40:00.000000000
Washington Wizards       1990-12-08 18:40:00.000000000
Charlotte Hornets        1990-11-01 08:50:31.578947328
Brooklyn Nets            1990-09-23 17:08:34.285714304
Chicago Bulls            1990-09-05 00:00:00.000000000
Oklahoma City Thunder    1990-09-04 12:37:53.684210560
Houston Rockets          1990-08-14 06:40:00.000000000
Utah Jazz                1990-04-26 19:12:00.000000000
Philadelphia 76ers       1990-03-25 12:00:00.000000000
Miami Heat               1989-11-12 19:12:00.000000000
Sacramento Kings         1989-11-05 07:34:44.210526336
New York Knicks          1989-10-21 22:30:00.000000000
New O

In [169]:
(pd.Timestamp.now() - df['birth_date']).dt.days

0      11767.0
2      12797.0
3      12797.0
4      11779.0
5      14614.0
        ...   
600    12067.0
601    13059.0
602    10632.0
603    11566.0
604    10442.0
Name: birth_date, Length: 551, dtype: float64

In [170]:
df['Age in days'] = (pd.Timestamp.now() - df['birth_date']).dt.days

In [171]:
df[['Player', 'birth_date', 'Age in days']].head()

,Player,birth_date,Age in days
0,Alex Abrines,1993-08-01,11767.0
2,Quincy Acy,1990-10-06,12797.0
3,Quincy Acy,1990-10-06,12797.0
4,Steven Adams,1993-07-20,11779.0
5,Arron Afflalo,1985-10-15,14614.0


In [172]:
df.groupby('Team')['Age in days'].mean().sort_values()

Team
Portland Trail Blazers    12271.666667
Toronto Raptors           12604.294118
Boston Celtics            12616.200000
Orlando Magic             12679.315789
Denver Nuggets            12696.263158
Detroit Pistons           12697.266667
Phoenix Suns              12709.888889
Washington Wizards        12733.222222
Charlotte Hornets         12770.631579
Brooklyn Nets             12809.285714
Chicago Bulls             12828.000000
Oklahoma City Thunder     12828.473684
Houston Rockets           12849.722222
Utah Jazz                 12959.200000
Philadelphia 76ers        12991.500000
Miami Heat                13124.200000
Sacramento Kings          13131.684211
New York Knicks           13146.062500
New Orleans Pelicans      13149.814815
Dallas Mavericks          13165.458333
Milwaukee Bucks           13233.350000
Memphis Grizzlies         13271.235294
Minnesota Timberwolves    13420.812500
Indiana Pacers            13557.882353
Golden State Warriors     13562.705882
Los Angeles Lakers  